# Load Model and Create Submission

## Overview

This notebook loads pre-trained model weights and generates predictions for the test set.

Useful for:
- Loading models trained in previous sessions
- Using pre-trained models from the model hub
- Generating submissions without retraining

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load test data
test_df = pd.read_csv('test.csv')
print(f"Test data loaded: {len(test_df)} samples")
print(test_df)

## 1. Load Model from Disk or Hub

In [ ]:
# Try loading from different sources in order of priority

MODEL_PATH = None
model = None

# Option 1: Local ensemble model
if os.path.exists("./ensemble_models/t5_small"):
    MODEL_PATH = "./ensemble_models/t5_small"
    print(f"Loading model from: {MODEL_PATH}")
    tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

# Option 2: Results directory
elif os.path.exists("./results_augmented/checkpoint-*"):
    import glob
    checkpoints = glob.glob("./results_augmented/checkpoint-*")
    if checkpoints:
        MODEL_PATH = sorted(checkpoints)[-1]  # Latest checkpoint
        print(f"Loading model from: {MODEL_PATH}")
        tokenizer = T5Tokenizer.from_pretrained("t5-small")
        model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

# Option 3: HuggingFace model hub (specify model name)
else:
    MODEL_NAME = "t5-small"  # Or specify your trained model name
    print(f"No local model found. Loading base {MODEL_NAME} from HuggingFace.")
    print("Note: This is a base model, not fine-tuned. Results may be suboptimal.")
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Move to device
model = model.to(DEVICE)
print(f"\nModel loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")

## 2. Generate Predictions

In [ ]:
print("Generating predictions...")

# Prepare inputs
test_texts = ["translate Akkadian to English: " + str(t) for t in test_df['transliteration'].tolist()]

# Tokenize
test_inputs = tokenizer(
    test_texts,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

# Move to device
test_inputs = {k: v.to(DEVICE) for k, v in test_inputs.items()}

# Generate
outputs = model.generate(
    input_ids=test_inputs['input_ids'],
    attention_mask=test_inputs['attention_mask'],
    max_length=128,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=3,
)

# Decode
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("\n--- Predictions ---")
for i, pred in enumerate(predictions):
    print(f"\nTest {i}:")
    print(f"  Input:    {test_df['transliteration'].iloc[i][:60]}...")
    print(f"  Output:   {pred[:100]}...")

## 3. Save Submission

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': predictions
})

# Save submission
submission.to_csv('submission_final.csv', index=False)
print("Submission saved to submission_final.csv")

print("\n--- Final Submission ---")
print(submission)

# Verify format
print("\n--- Verification ---")
sample = pd.read_csv('sample_submission.csv')
print(f"Sample columns: {sample.columns.tolist()}")
print(f"Our columns:    {submission.columns.tolist()}")
print(f"Match: {list(sample.columns) == list(submission.columns)}")

## 4. Also Generate with Different Decoding Strategies

In [ ]:
# Try different decoding strategies
decoding_strategies = {
    'greedy': {'num_beams': 1},
    'beam': {'num_beams': 4, 'early_stopping': True},
    'beam_ngram': {'num_beams': 4, 'no_repeat_ngram_size': 3, 'early_stopping': True},
}

all_predictions = {}

for name, params in decoding_strategies.items():
    outputs = model.generate(
        input_ids=test_inputs['input_ids'],
        attention_mask=test_inputs['attention_mask'],
        max_length=128,
        **params
    )
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_predictions[name] = preds
    print(f"\n{name} strategy:")
    for i, pred in enumerate(preds[:2]):
        print(f"  {i}: {pred[:60]}...")

## 5. Summary

In [ ]:
print("=" * 60)
print("SUBMISSION SUMMARY")
print("=" * 60)

print(f"""
Model source: {MODEL_PATH or 't5-small (base)'}
Device used: {DEVICE}
Test samples: {len(test_df)}
Output file: submission_final.csv

Submission format verified against sample_submission.csv
""")

print("Files created:")
print("  - submission_final.csv: Final submission file")